In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install trl==0.12.0  # or whatever version your tutorial expects

!pip install -U bitsandbytes>=0.46.1

In [ ]:
# ============================================================
# QWEN 2.5-CODER 1.5B
# QLoRA SFT — WINDOWS MALWARE CLASSIFICATION
#
# FINAL CORRECTED VERSION
#
# Dataset: 17,693 training samples
# GPU: Single Tesla T4
# Epochs: 1
# Max sequence length: 2048
# ============================================================


# ============================================================
# 1. GPU CONFIGURATION
# MUST BE BEFORE IMPORTING TORCH
# ============================================================

import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


# ============================================================
# 2. IMPORTS
# ============================================================

import torch

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
)

from peft import (
    LoraConfig,
    prepare_model_for_kbit_training,
)

from trl import (
    SFTConfig,
    SFTTrainer,
)


# ============================================================
# 3. ENVIRONMENT
# ============================================================

print("=" * 70)
print("ENVIRONMENT")
print("=" * 70)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3,
            2
        ),
        "GB"
    )

print("=" * 70)


# ============================================================
# 4. PATHS
# ============================================================

TRAIN_FILE = (
    "/kaggle/input/datasets/mohdtahasayed2904/taha-12234/LLM_SFT_Train_17693.jsonl"
)

MODEL_NAME = (
    "Qwen/Qwen2.5-Coder-1.5B-Instruct"
)

OUTPUT_DIR = (
    "/kaggle/working/"
    "Qwen2.5-Coder-1.5B-QLoRA-SFT"
)

FINAL_ADAPTER_DIR = os.path.join(
    OUTPUT_DIR,
    "final_adapter"
)

MAX_LENGTH = 2048


# ============================================================
# 5. VERIFY TRAINING FILE
# ============================================================

print("\n" + "=" * 70)
print("VERIFYING TRAINING DATA")
print("=" * 70)

assert os.path.exists(TRAIN_FILE), (
    f"Training file not found:\n{TRAIN_FILE}"
)

print(
    "Training file:",
    TRAIN_FILE
)

print(
    "File size:",
    round(
        os.path.getsize(TRAIN_FILE) / 1024**2,
        2
    ),
    "MB"
)


# ============================================================
# 6. LOAD DATASET
# ============================================================

dataset = load_dataset(
    "json",
    data_files=TRAIN_FILE,
    split="train"
)

print("\nDataset:")
print(dataset)

print(
    "Number of records:",
    len(dataset)
)


# ============================================================
# 7. CHECK DATA
# ============================================================

print("\n" + "=" * 70)
print("CHECKING DATA FORMAT")
print("=" * 70)

print(
    "Columns:",
    dataset.column_names
)

first_record = dataset[0]

print(
    "\nFirst record keys:"
)

print(
    first_record.keys()
)

print(
    "\nMessage roles:"
)

print([
    message["role"]
    for message in first_record["messages"]
])

print(
    "\nLabel:",
    first_record["label"]
)

print(
    "Label ID:",
    first_record["label_id"]
)


# ============================================================
# 8. TOKENIZER
# ============================================================

print("\n" + "=" * 70)
print("LOADING TOKENIZER")
print("=" * 70)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print(
    "Tokenizer loaded"
)

print(
    "Vocab size:",
    len(tokenizer)
)

print(
    "EOS token:",
    tokenizer.eos_token
)

print(
    "EOS ID:",
    tokenizer.eos_token_id
)

print(
    "PAD token:",
    tokenizer.pad_token
)

print(
    "PAD ID:",
    tokenizer.pad_token_id
)


# ============================================================
# 9. APPLY QWEN CHAT TEMPLATE
# ============================================================

print("\n" + "=" * 70)
print("FORMATTING CHAT DATA")
print("=" * 70)


def format_chat(example):

    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )

    return {
        "text": text
    }


formatted_dataset = dataset.map(
    format_chat,
    remove_columns=dataset.column_names,
    desc="Formatting chat templates"
)

print(
    "Formatted dataset:"
)

print(
    formatted_dataset
)

print(
    "Number of examples:",
    len(formatted_dataset)
)


# ============================================================
# 10. TOKENIZE DATASET
#
# THIS FIXES THE ERROR YOU JUST GOT.
#
# The previous trainer was receiving:
#     text = "...."
#
# instead of:
#     input_ids = [...]
#     attention_mask = [...]
#
# We explicitly create those fields here.
# ============================================================

print("\n" + "=" * 70)
print("TOKENIZING TRAINING DATA")
print("=" * 70)


def tokenize_function(example):

    return tokenizer(
        example["text"],

        # Truncate long examples
        truncation=True,

        # Pad every example to a fixed length
        padding="max_length",

        # Maximum context
        max_length=MAX_LENGTH,
    )


tokenized_dataset = formatted_dataset.map(
    tokenize_function,

    batched=True,

    # We no longer need raw text
    remove_columns=["text"],

    desc="Tokenizing training data"
)


# ============================================================
# 11. CHECK TOKENIZED DATA
# ============================================================

print(
    "\nTokenized dataset:"
)

print(
    tokenized_dataset
)

print(
    "\nColumns:"
)

print(
    tokenized_dataset.column_names
)

print(
    "\nFirst tokenized example:"
)

print(
    tokenized_dataset[0].keys()
)

print(
    "input_ids length:",
    len(tokenized_dataset[0]["input_ids"])
)

print(
    "attention_mask length:",
    len(tokenized_dataset[0]["attention_mask"])
)

assert (
    len(tokenized_dataset[0]["input_ids"])
    == MAX_LENGTH
)

assert (
    len(tokenized_dataset[0]["attention_mask"])
    == MAX_LENGTH
)

print(
    "\nTokenization check PASSED"
)


# ============================================================
# 12. 4-BIT QLoRA CONFIG
# ============================================================

print("\n" + "=" * 70)
print("CONFIGURING 4-BIT QLoRA")
print("=" * 70)

bnb_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_compute_dtype=torch.bfloat16,

    bnb_4bit_use_double_quant=True,
)

print(
    "4-bit NF4 configuration ready"
)


# ============================================================
# 13. LOAD BASE MODEL
# ============================================================

print("\n" + "=" * 70)
print("LOADING QWEN MODEL")
print("=" * 70)

model = AutoModelForCausalLM.from_pretrained(

    MODEL_NAME,

    quantization_config=bnb_config,

    # Single T4
    device_map={
        "": 0
    },

    dtype=torch.bfloat16,

    trust_remote_code=True,
)

print(
    "Base model loaded"
)


# ============================================================
# 14. PREPARE MODEL FOR K-BIT TRAINING
# ============================================================

model = prepare_model_for_kbit_training(
    model
)

model.config.use_cache = False


# ============================================================
# 15. LoRA CONFIGURATION
# ============================================================

print("\n" + "=" * 70)
print("CONFIGURING LoRA")
print("=" * 70)

lora_config = LoraConfig(

    r=16,

    lora_alpha=32,

    lora_dropout=0.05,

    bias="none",

    task_type="CAUSAL_LM",

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

print(
    lora_config
)


# ============================================================
# 16. DATA COLLATOR
# ============================================================

print("\n" + "=" * 70)
print("CREATING DATA COLLATOR")
print("=" * 70)

data_collator = DataCollatorForLanguageModeling(

    tokenizer=tokenizer,

    mlm=False,
)

print(
    "Causal-LM data collator created"
)


# ============================================================
# 17. SFT CONFIGURATION
# ============================================================

print("\n" + "=" * 70)
print("CREATING SFT CONFIG")
print("=" * 70)

sft_config = SFTConfig(

    # --------------------------------------------------------
    # OUTPUT
    # --------------------------------------------------------

    output_dir=OUTPUT_DIR,


    # --------------------------------------------------------
    # TRAINING
    # --------------------------------------------------------

    num_train_epochs=1,

    per_device_train_batch_size=1,

    gradient_accumulation_steps=16,


    # --------------------------------------------------------
    # SEQUENCE LENGTH
    # --------------------------------------------------------

    max_seq_length=MAX_LENGTH,


    # --------------------------------------------------------
    # LEARNING RATE
    # --------------------------------------------------------

    learning_rate=2e-4,

    weight_decay=0.01,


    # --------------------------------------------------------
    # OPTIMIZER
    # --------------------------------------------------------

    optim="paged_adamw_8bit",


    # --------------------------------------------------------
    # SCHEDULER
    # --------------------------------------------------------

    warmup_steps=33,

    lr_scheduler_type="cosine",


    # --------------------------------------------------------
    # LOGGING
    # --------------------------------------------------------

    logging_strategy="steps",

    logging_steps=50,


    # --------------------------------------------------------
    # CHECKPOINTS
    # --------------------------------------------------------

    save_strategy="steps",

    save_steps=50,

    save_total_limit=3,


    # --------------------------------------------------------
    # PRECISION
    # --------------------------------------------------------

    bf16=(
        torch.cuda.is_available()
        and torch.cuda.is_bf16_supported()
    ),

    fp16=(
        torch.cuda.is_available()
        and not torch.cuda.is_bf16_supported()
    ),


    # --------------------------------------------------------
    # GRADIENT CHECKPOINTING
    # --------------------------------------------------------

    gradient_checkpointing=True,

    gradient_checkpointing_kwargs={
        "use_reentrant": False
    },


    # --------------------------------------------------------
    # DATALOADER
    # --------------------------------------------------------

    dataloader_num_workers=0,


    # --------------------------------------------------------
    # REPRODUCIBILITY
    # --------------------------------------------------------

    seed=42,

    data_seed=42,


    # --------------------------------------------------------
    # IMPORTANT:
    #
    # We are providing input_ids and attention_mask
    # ourselves, so don't remove them.
    # --------------------------------------------------------

    remove_unused_columns=False,


    # --------------------------------------------------------
    # REPORTING
    # --------------------------------------------------------

    report_to="none",
)

print(
    "SFTConfig created successfully"
)


# ============================================================
# 18. CREATE SFT TRAINER
# ============================================================

print("\n" + "=" * 70)
print("CREATING SFT TRAINER")
print("=" * 70)

trainer = SFTTrainer(

    model=model,

    args=sft_config,

    train_dataset=tokenized_dataset,

    processing_class=tokenizer,

    data_collator=data_collator,

    peft_config=lora_config,
)

print(
    "SFTTrainer created successfully"
)


# ============================================================
# 19. TRAINABLE PARAMETERS
# ============================================================

print("\n" + "=" * 70)
print("TRAINABLE PARAMETERS")
print("=" * 70)

trainer.model.print_trainable_parameters()


# ============================================================
# 20. TRAINING CONFIGURATION
# ============================================================

print("\n" + "=" * 70)
print("TRAINING CONFIGURATION")
print("=" * 70)

print(
    "Base model:",
    MODEL_NAME
)

print(
    "Training samples:",
    len(tokenized_dataset)
)

print(
    "Epochs:",
    1
)

print(
    "Max sequence length:",
    MAX_LENGTH
)

print(
    "Per-device batch size:",
    1
)

print(
    "Gradient accumulation:",
    16
)

print(
    "Effective batch size:",
    16
)

print(
    "Learning rate:",
    2e-4
)

print(
    "Weight decay:",
    0.01
)

print(
    "Warmup steps:",
    33
)

print(
    "Optimizer:",
    "paged_adamw_8bit"
)

print(
    "LoRA rank:",
    16
)

print(
    "LoRA alpha:",
    32
)

print(
    "GPU:",
    torch.cuda.get_device_name(0)
)

print("=" * 70)


# ============================================================
# 21. START TRAINING
# ============================================================

print("\n" + "=" * 70)
print("STARTING QLoRA SFT TRAINING")
print("=" * 70)

train_result = trainer.train()


# ============================================================
# 22. TRAINING RESULTS
# ============================================================

print("\n" + "=" * 70)
print("TRAINING COMPLETED")
print("=" * 70)

print(
    "Training result:"
)

print(
    train_result
)

if hasattr(
    train_result,
    "metrics"
):

    print(
        "\nTraining metrics:"
    )

    for key, value in train_result.metrics.items():

        print(
            f"{key}: {value}"
        )


# ============================================================
# 23. SAVE FINAL LoRA ADAPTER
# ============================================================

print("\n" + "=" * 70)
print("SAVING FINAL LoRA ADAPTER")
print("=" * 70)

os.makedirs(
    FINAL_ADAPTER_DIR,
    exist_ok=True
)

trainer.model.save_pretrained(
    FINAL_ADAPTER_DIR
)

tokenizer.save_pretrained(
    FINAL_ADAPTER_DIR
)

print(
    "Final adapter saved to:"
)

print(
    FINAL_ADAPTER_DIR
)


# ============================================================
# 24. VERIFY SAVED ADAPTER
# ============================================================

print("\n" + "=" * 70)
print("VERIFYING SAVED ADAPTER")
print("=" * 70)

for root, dirs, files in os.walk(
    FINAL_ADAPTER_DIR
):

    for file in files:

        filepath = os.path.join(
            root,
            file
        )

        size_mb = (
            os.path.getsize(filepath)
            / 1024**2
        )

        print(
            f"{file:<45}"
            f"{size_mb:.2f} MB"
        )


# ============================================================
# 25. FINAL GPU MEMORY
# ============================================================

if torch.cuda.is_available():

    print("\n" + "=" * 70)
    print("FINAL GPU MEMORY")
    print("=" * 70)

    allocated = (
        torch.cuda.memory_allocated(0)
        / 1024**3
    )

    reserved = (
        torch.cuda.memory_reserved(0)
        / 1024**3
    )

    print(
        f"Allocated: {allocated:.2f} GB"
    )

    print(
        f"Reserved:  {reserved:.2f} GB"
    )


# ============================================================
# 26. FINAL STATUS
# ============================================================

print("\n" + "=" * 70)
print("QLoRA SFT COMPLETE")
print("=" * 70)

print(
    "Adapter:",
    FINAL_ADAPTER_DIR
)

print(
    "Training samples:",
    len(tokenized_dataset)
)

print(
    "Epochs:",
    1
)

print(
    "Sequence length:",
    MAX_LENGTH
)

print(
    "Status: SUCCESS"
)

print("=" * 70)

In [ ]:
import os

OUTPUT_DIR = "/kaggle/working/Qwen2.5-Coder-1.5B-QLoRA-SFT"

print("=" * 70)
print("CHECKING AVAILABLE CHECKPOINTS")
print("=" * 70)

if os.path.exists(OUTPUT_DIR):
    for item in sorted(os.listdir(OUTPUT_DIR)):
        path = os.path.join(OUTPUT_DIR, item)

        if os.path.isdir(path):
            print(f"\n📁 {item}/")

            for f in os.listdir(path):
                fp = os.path.join(path, f)
                size_mb = os.path.getsize(fp) / 1024**2

                print(f"   {f:<40} {size_mb:.2f} MB")
else:
    print("Output directory does not exist yet.")